# 학습: 과거 최고 앙상블의 중첩 Scaffold 검증

기존 최고 모델 계열은 고정하고 가중치와 임계값은 안쪽 CV에서만 선택한다. 안쪽 선택과 바깥 평가는 모두 5-Fold로 실행한다. 바깥 Scaffold Fold는 전처리 학습과 선택에 사용하지 않는다.

원본 label: 0=독성, 1=비독성. 주지표는 비독성 F1, 보조지표는 독성 Recall이다. 기존 전체 데이터 EDA와 모델 계열 선택 이력이 있어 독립 외부 검증은 아니다.

`python -m pip install -r requirements-training.txt` 후 프로젝트 루트에서 실행한다. 출력 폴더가 이미 있으면 덮어쓰지 않으며 새 폴더를 지정한다.

기준선 목표: 독성 Recall ≥ 0.90인 안쪽 OOF 임계값 중 비독성 F1 최대화. 바깥 평가에서 비독성 F1 ≥ 0.85, 독성 Recall ≥ 0.90 달성 여부를 확인한다. 임계값은 0~1을 0.01 간격으로 탐색하며 전체 독성 판정 끝점을 포함한다.


In [ ]:
from pathlib import Path
import json, hashlib
from analysis.train_safe import run

# 새 학습은 빈 OUTPUT 경로를 지정한다. 완료된 실행은 재학습 없이 읽는다.
OUTPUT = Path('Data/training/recall90_baseline_5fold')
if (OUTPUT / 'evaluation.json').is_file():
    config = json.loads((OUTPUT / 'run_config.json').read_text())
    assert hashlib.sha256(Path('Data/input_data/train.csv').read_bytes()).hexdigest() == config['source_sha256']
    assert config['recall_target'] == 0.90 and config['outer_folds'] == 5 and config['inner_folds'] == 5
    result = json.loads((OUTPUT / 'evaluation.json').read_text())
    print('완료된 실행 결과를 읽었습니다. 새 학습은 OUTPUT을 새 폴더로 바꾸세요.')
else:
    result = run(output=OUTPUT, recall_target=0.90)

## 결과 확인
바깥 평가는 선택 과정의 내부 검증이다. 마지막 전체 재학습 모델은 배포용이며 별도의 성능 점수를 얻은 것이 아니다. 점수를 본 뒤 이번 실행의 설정이나 임계값을 바꾸지 않는다.

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(OUTPUT / 'outer_metrics.csv'))
print(result)